[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/solutions/38_cross_entropy_solution.ipynb)

# 🟢 Solution: Cross-Entropy Loss

*Training · Easy*

Reference implementation. Try it yourself in `38_cross_entropy.ipynb` first.

---
Implement **cross-entropy loss directly from logits**, with optional label
smoothing and a padding mask.

$$\ell_i = -\sum_{c} q_{i,c} \log p_{i,c}, \qquad
p_{i,c} = \frac{e^{z_{i,c}}}{\sum_{k} e^{z_{i,k}}}$$

With smoothing $\alpha$ over $C$ classes the target distribution is
$q_{i,c} = (1-\alpha)\,\mathbb{1}[c = t_i] + \alpha/C$, so

$$\ell_i = (1-\alpha)\bigl(-\log p_{i,t_i}\bigr) \;+\; \frac{\alpha}{C}\sum_c \bigl(-\log p_{i,c}\bigr)$$

Return the **mean over the non-ignored positions only**.

### Rules
- Signature: `cross_entropy_loss(logits, targets, *, label_smoothing=0.0, ignore_index=-1)`
- `logits` is `(..., C)`, `targets` is `(...)` of integer class ids; the output is a **scalar**
- Banned: `jax.nn.log_softmax`, `jax.nn.softmax`, `jax.scipy.special.logsumexp`, `optax`
- Compute $\log p$ in one fused expression; never form $p$ and then take its log
- Positions where `targets == ignore_index` contribute **zero** loss and are excluded
  from the denominator; if every position is ignored, return `0.0` (not `nan`)
- No `if` on array values — the whole thing must work under `jit` and `vmap`

### Why you never softmax-then-log
The naive route dies twice.

**Overflow.** `exp(z)` is `inf` above $z \approx 88.7$ in float32 and above
$z \approx 11.1$ in float16. The fix is the shift
$\log \sum_k e^{z_k} = m + \log \sum_k e^{z_k - m}$ with $m = \max_k z_k$: every
exponent is now $\le 0$, so the largest term is exactly `1.0` and the sum can
never overflow.

**Underflow — the one that actually bites.** Even with no overflow, a confidently
*wrong* prediction pushes $p_t$ under the float32 floor: normals stop at
$\approx 1.2\times10^{-38}$ and subnormals at $\approx 1.4\times10^{-45}$, and
XLA flushes subnormals to zero on accelerators anyway. Once $p_t$ rounds to
`0.0`, `log(0) = -inf` makes the loss `inf` and every gradient `nan`. The fused
form never materialises $p_t$: it computes $z_t - \log\sum_k e^{z_k}$, a
perfectly finite number like $-120$ (that is $p_t \approx 10^{-52}$, hopelessly
unrepresentable, yet its logarithm is an ordinary float). Your loss stays
large-but-finite and training recovers instead of poisoning every parameter
with `nan`.

There is a gradient bonus too. $\partial \ell / \partial z = p - q$ — a clean,
bounded expression that autodiff derives exactly from the fused form. Compose
`log` on top of a separate `softmax` and you hand XLA a division of two tiny
numbers to differentiate through.

The `max` shift cancels analytically ($\ell$ is invariant to it), so wrapping it
in `stop_gradient` changes nothing mathematically and keeps the backward graph
smaller.

### The interview angle
`ignore_index` is where candidates lose the plot. Pad-to-longest batching over
variable-length sequences routinely leaves a third or more of the positions as
`<pad>`. Average over all of them and two things go wrong: the loss is scaled
down by the pad fraction (so it silently changes meaning when the batch
composition changes), and the gradient actively teaches the model to predict
`<pad>`.

The second half is the index itself, and it is worth knowing exactly what JAX
does here because it does **not** raise. `jnp.take_along_axis` defaults to
`mode="fill"`, so a genuinely out-of-range sentinel like `-100` gathers `NaN` —
which then propagates through the mean and poisons the whole batch even though
you "masked" it afterwards. And `-1` does not go out of range at all: it wraps
round to the last class, so you get a plausible-looking wrong loss with nothing
to alert you. Both are fixed the same way — substitute a valid index *before*
the gather — but only if you knew there was something to fix.

In [ ]:
# Install jax-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q jax-judge flax')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✅ REFERENCE SOLUTION

import jax
import jax.numpy as jnp


def cross_entropy_loss(logits, targets, *, label_smoothing=0.0, ignore_index=-1):
    logits = jnp.asarray(logits)
    targets = jnp.asarray(targets)
    num_classes = logits.shape[-1]

    # log-softmax, fused. Shifting by the row max makes every exponent <= 0, so
    # the sum is in [1, C] and can never overflow. The shift cancels exactly in
    # the final expression, hence stop_gradient.
    shift = jax.lax.stop_gradient(jnp.max(logits, axis=-1, keepdims=True))
    z = logits - shift
    log_probs = z - jnp.log(jnp.sum(jnp.exp(z), axis=-1, keepdims=True))

    valid = targets != ignore_index
    # Clamp BEFORE gathering: JAX silently clips out-of-range indices.
    safe = jnp.where(valid, targets, 0)
    nll = -jnp.take_along_axis(log_probs, safe[..., None], axis=-1)[..., 0]

    # Smoothing is a convex blend with the uniform target: (a/C) * sum_c -log p_c.
    uniform = -jnp.mean(log_probs, axis=-1)
    per_position = (1.0 - label_smoothing) * nll + label_smoothing * uniform

    per_position = jnp.where(valid, per_position, 0.0)
    n_valid = jnp.sum(valid)
    return jnp.sum(per_position) / jnp.maximum(n_valid, 1)

In [ ]:
# 🔍 Verify
import jax
import jax.numpy as jnp

# Uniform logits over 3 classes -> loss is exactly log(3).
print("uniform:", cross_entropy_loss(jnp.zeros((1, 3)), jnp.array([0])), "vs", jnp.log(3.0))

# The stability trap: huge logits.
big = jnp.array([[1000.0, 0.0, 0.0]])
print("big logits, correct class:", cross_entropy_loss(big, jnp.array([0])))
print("naive softmax-then-log would give:",
      -jnp.log(jnp.exp(big) / jnp.exp(big).sum(-1, keepdims=True))[0, 0])

# Padding. Non-uniform logits, so the denominator genuinely matters.
logits = jax.random.normal(jax.random.key(0), (4, 5)) * 3.0
print("\nmean over the 2 real tokens:",
      float(cross_entropy_loss(logits, jnp.array([1, 2, -1, -1]), ignore_index=-1)))
print("mean over all 4 positions  :",
      float(cross_entropy_loss(logits, jnp.array([1, 2, 0, 0]))), " <- wrong denominator")

# Why the index must be sanitised BEFORE the gather: JAX never raises here.
print("\ngather at -1  ->", jnp.take_along_axis(logits, jnp.full((4, 1), -1), -1)[:, 0])
print("               ...that is column 4, read silently")
print("gather at -100->", jnp.take_along_axis(logits, jnp.full((4, 1), -100), -1)[:, 0])
print("               ...NaN, which masking afterwards will not remove")

# Smoothing penalises over-confidence.
sharp, t0 = jnp.array([[20.0, 0.0, 0.0]]), jnp.array([0])
print("\nsharp, alpha=0.0:", float(cross_entropy_loss(sharp, t0)))
print("sharp, alpha=0.1:", float(cross_entropy_loss(sharp, t0, label_smoothing=0.1)))

In [ ]:
# Run the judge against the reference solution
from jax_judge import check

check("cross_entropy")